In [37]:
from langgraph.graph import StateGraph, START, END
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from dotenv import load_dotenv
from typing import TypedDict
import os

In [38]:
load_dotenv()

token = os.getenv("HUGGINGFACEHUB_API_TOKEN") or os.getenv("HUGGINGFACEHUB_ACCESS_TOKEN")
if not token:
    raise ValueError("Missing Hugging Face token in environment or .env file")

os.environ.setdefault("HUGGINGFACEHUB_API_TOKEN", token)

llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-72B-Instruct",
    task="text-generation",
    huggingfacehub_api_token=token,
)

model = ChatHuggingFace(llm=llm)

In [39]:
class BlogState(TypedDict):
    topic: str
    outline: str
    content: str

In [40]:
def generate_outline(state: BlogState) -> BlogState:
    topic = state["topic"]
    prompt = f"Write a detailed outline for a blog post about {topic}."
    outline = model.invoke(prompt).content
    state["outline"] = outline
    return state

In [41]:
def generate_content(state: BlogState) -> BlogState:
    outline = state["outline"]
    prompt = f"Write a detailed blog post based on the following outline:\n{outline}"
    content = model.invoke(prompt).content
    state["content"] = content
    return state

In [42]:
graph = StateGraph(BlogState)

In [43]:
graph.add_node("generate_outline", generate_outline)
graph.add_node("generate_content", generate_content)

graph.add_edge(START, "generate_outline")
graph.add_edge("generate_outline", "generate_content")
graph.add_edge("generate_content", END)

In [44]:
workflow = graph.compile()

In [45]:
initial_state = {
    "topic": "Python programming",
    "outline": "",
    "content": "",
}

In [46]:
final_state = workflow.invoke(initial_state)

In [47]:
print(final_state)

{'topic': 'Python programming', 'outline': 'Certainly! Below is a detailed outline for a blog post about Python programming. This outline covers a range of topics to provide a comprehensive introduction and guide for both beginners and intermediate learners.\n\n### Title: Mastering Python Programming: A Comprehensive Guide\n\n### Introduction\n- Brief overview of Python programming\n- Importance and popularity of Python in various fields (data science, web development, automation, etc.)\n- Target audience: beginners and intermediate learners\n\n### Section 1: Getting Started with Python\n#### 1.1. What is Python?\n- Definition and history of Python\n- Key features: simplicity, readability, and versatility\n\n#### 1.2. Why Choose Python?\n- Ease of learning and use\n- Large community and extensive resources\n- Cross-platform compatibility (Windows, macOS, Linux)\n\n#### 1.3. Installing Python\n- Downloading and installing Python from the official website\n- Setting up the environment (I